# WAV1 mechanism factorization — HP1
Satu notebook = satu model. Notebook ini hanya menjalankan **HP1 seed 42** dan aman diparalelkan dengan notebook arm lain pada akun Colab berbeda. Semua akun membaca satu shared `Coffee_Bean_Detection`, tetapi output HP1 ditulis ke folder khusus sehingga tidak bertabrakan. Locked test tetap tertutup.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import importlib, json, os, shutil, subprocess, sys, tarfile, time, torch
from pathlib import Path
assert torch.cuda.is_available(), 'Aktifkan GPU di Runtime → Change runtime type.'

ARM='HP1'
BRANCH='agent/wav1-mechanism-factorization'
REPO=Path('/content/coffee-bean-detection')
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)

from coffee_detector.drive_project import resolve_drive_project_root, require_project_artifact
from coffee_detector.wav1_factorization.audit import run_static_audit
from coffee_detector.wav1_factorization import WAV1FactorizationEnhancer, frozen_arm_config

REQ=(
  'bundles/faruq-development-v3-grouped.tar',
  'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt',
)
PROJECT=resolve_drive_project_root(required_relative_paths=REQ)
ARCHIVE=require_project_artifact(PROJECT,REQ[0])
D0=require_project_artifact(PROJECT,REQ[1])
DATA=Path('/content/faruq-development-v3-grouped')
if not (DATA/'data.yaml').is_file():
    with tarfile.open(ARCHIVE,'r') as archive: archive.extractall('/content',filter='data')
assert (DATA/'data.yaml').is_file() and (DATA/'faruq_grouped_summary.json').is_file()
assert not (DATA/'test').exists(), 'STOP: development dataset mengekspos test.'

BASE=PROJECT/'experiments/faruq-v3-wav1-mechanism-factorization-v1/parallel'
OUT=BASE/ARM; OUT.mkdir(parents=True,exist_ok=True)
STATIC=OUT/'static_audit.json'
audit=run_static_audit(D0,STATIC)
assert audit['decision']=='PASS' and audit['training_authorized'] is True
assert audit['wav1_ref_bitwise_equal_to_confirmed_operator'] is True
assert audit['test_access_authorized'] is False

previous=torch.are_deterministic_algorithms_enabled()
try:
    torch.use_deterministic_algorithms(True,warn_only=False)
    probe=torch.rand(1,3,64,64,device='cuda:0',requires_grad=True)
    frontend=WAV1FactorizationEnhancer(frozen_arm_config(ARM)).to('cuda:0')
    value=frontend(probe); value.mean().backward()
    assert torch.isfinite(value).all() and probe.grad is not None and torch.isfinite(probe.grad).all()
finally:
    torch.use_deterministic_algorithms(previous)

print('GPU:',torch.cuda.get_device_name(0))
print('PROJECT:',PROJECT)
print('ARM:',ARM)
print('OUTPUT:',OUT)
print('STATIC AUDIT: PASS | CUDA SMOKE: PASS | TEST LOCK: CLOSED')


In [ ]:
RESULT=OUT/'val_reports'/f'{ARM}_seed42_result.json'
LOG=OUT/f'{ARM}_seed42_run.log'
if RESULT.is_file():
    print('REUSE COMPLETE:',RESULT)
else:
    cmd=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_wav1_factorization_arm',
         '--arm',ARM,'--data-root',str(DATA),'--grouped-summary',str(DATA/'faruq_grouped_summary.json'),
         '--d0-checkpoint',str(D0),'--static-audit',str(STATIC),'--output-root',str(OUT),
         '--seed','42','--device','0','--authorize-training']
    print('MENJALANKAN:', ' '.join(cmd),flush=True)
    with LOG.open('a',encoding='utf-8') as handle:
        process=subprocess.Popen(cmd,cwd=REPO,stdout=handle,stderr=subprocess.STDOUT)
    last=-1
    while process.poll() is None:
        csv=OUT/ARM/f'{ARM}_seed42'/'results.csv'
        epoch=max(0,len(csv.read_text(errors='replace').splitlines())-1) if csv.is_file() else 0
        if epoch!=last: print(f'{ARM}: {epoch}/50 epoch | log={LOG}',flush=True); last=epoch
        time.sleep(180)
    if process.returncode:
        tail='\n'.join(LOG.read_text(errors='replace').splitlines()[-160:]) if LOG.is_file() else '<log tidak ada>'
        raise RuntimeError(f'{ARM} gagal: returncode={process.returncode}\n--- LOG TAIL ---\n{tail}')
assert RESULT.is_file(), f'Result tidak ditemukan: {RESULT}'
result=json.loads(RESULT.read_text(encoding='utf-8'))
assert result['evaluation_split']=='val' and result['test_images_accessed'] is False
m=result['metrics']
print('\nSELESAI',ARM)
print('Macro:',m['macro_map50_95'],'Bottom3:',m['bottom3_class_map50_95'],'Worst:',m['worst_class_map50_95'])
print('RESULT:',RESULT)
print('Output sudah langsung tersimpan di shared Drive; tidak perlu download checkpoint.')
